# Session 6: Model Evaluation and Generalization
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/buildLittleWorlds/level-2-course-material/blob/main/session-06/notebook.ipynb)

Session 5 gave us controls — temperature, top-p, max length. Same knobs on ChatGPT and Claude. Session 6 asks: **what are the limits?** Every model only knows what it was trained on. Tonight we take generative models out of their training world and watch them struggle.

In [ ]:
# Setup — run this cell first!
!pip install -q transformers torch

from transformers import pipeline

print("Loading text generator (distilgpt2)...")
generator = pipeline("text-generation", model="distilbert/distilgpt2")

print("Loading summarizer (distilbart-cnn)...")
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6")

print("Both models loaded!")

## What We Built Tonight

We built a **domain-adapted text generator** — same distilgpt2 model from Session 5 but with prompt presets for different domains (game dialogue, medical notes, news, poetry). We also tested the **summarizer** from Session 5 across different kinds of text.

The discovery: **same model, different domains, different quality.** The text generator was trained on web text. It writes decent game dialogue but terrible medical notes. The summarizer was trained on CNN/DailyMail news articles. It summarizes news well but loses meaning on poetry. That's **domain shift** — the model didn't get dumber, the world changed around it.

## Helper Functions

In [ ]:
def generate_from_prompt(prompt, temperature=0.7, max_new_tokens=80):
    """Generate text from a prompt and show the result."""
    result = generator(prompt, max_new_tokens=max_new_tokens,
                       temperature=max(temperature, 0.01),
                       do_sample=True, truncation=True)
    text = result[0]["generated_text"]
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {text}")
    print()
    return text


def summarize_text(text, max_length=100, min_length=25):
    """Summarize text and show original vs. summary word counts."""
    word_count = len(text.split())
    result = summarizer(text[:1024], max_length=max_length,
                        min_length=min(min_length, max_length - 10),
                        do_sample=False)
    summary = result[0]["summary_text"]
    summary_words = len(summary.split())
    print(f"ORIGINAL ({word_count} words):")
    print(f"  {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"SUMMARY ({summary_words} words):")
    print(f"  {summary}")
    print()
    return summary

---
## Part 1: Text Generator Across Domains

The text generator (distilgpt2) was trained on **WebText** — text scraped from links shared on Reddit. Let's see what happens when we give it prompts from different worlds.

### Domain: Game Dialogue
Does the model know what game writing sounds like?

In [ ]:
generate_from_prompt("The warrior stepped into the dungeon and said:")

### Domain: Medical Notes
Clinical language is very different from web text. How does the model handle it?

In [ ]:
generate_from_prompt("Patient presents with acute onset of substernal chest pain radiating to the left arm.")

### Domain: News Article
News is closer to web text — the model may handle this better.

In [ ]:
generate_from_prompt("Breaking news: scientists announced today that")

### Domain: Poetry
Poetry uses language differently — imagery, rhythm, compressed meaning.

In [ ]:
generate_from_prompt("I wandered through the silver rain and found")

### Domain: Legal Text
Legal language is formal, precise, and very different from casual web text.

In [ ]:
generate_from_prompt("The party of the first part shall indemnify and hold harmless")

---
## Part 2: Summarizer Across Domains

The summarizer (distilbart-cnn) was trained on **CNN/DailyMail news articles**. It should be great at news — but what about everything else?

### Domain: News Article (Home Turf)

In [ ]:
news = """Artificial intelligence has transformed many industries over the past decade. 
In healthcare, AI systems can now detect diseases from medical images with accuracy 
rivaling human doctors. In finance, algorithmic trading powered by machine learning 
processes millions of transactions per second. Education is also being reshaped, with 
AI tutors providing personalized learning experiences for students around the world. 
However, these advances come with significant challenges. Privacy concerns arise when 
AI systems require vast amounts of personal data. Job displacement remains a worry as 
automation replaces routine tasks. And bias in AI systems can perpetuate or even amplify 
existing social inequalities."""

summarize_text(news)

### Domain: Poetry

In [ ]:
poem = """I wandered lonely through the ash of things that used to gleam. The world 
had shed its golden mask and left a hollow dream. The rivers spoke in ancient tongues 
of sorrow and of time, while mountains held their heavy breath beneath the silver rime. 
I searched for something left to hold, a warmth against the cold, but every door I 
opened led to stories never told. The stars above were witnesses to everything we lost, 
and still they burned indifferent to the overwhelming cost."""

summarize_text(poem)

### Domain: Legal Text

In [ ]:
legal = """The party of the first part shall indemnify and hold harmless the party of 
the second part against any and all claims, damages, losses, costs, and expenses 
arising out of or relating to any breach of this agreement. In the event of a dispute 
between the parties, such dispute shall be resolved through binding arbitration in 
accordance with the rules of the American Arbitration Association. Neither party shall 
be liable for any indirect, incidental, special, consequential, or punitive damages 
arising out of this agreement, regardless of the theory of liability."""

summarize_text(legal)

### Domain: Medical Report

In [ ]:
medical = """The patient presented to the emergency department with acute onset of 
substernal chest pain radiating to the left arm, accompanied by diaphoresis and 
shortness of breath. Initial ECG showed ST-segment elevation in leads II, III, and 
aVF, consistent with inferior myocardial infarction. Troponin levels were elevated 
at 2.4 ng/mL. The patient was started on dual antiplatelet therapy and heparin 
infusion, and cardiology was consulted for emergent cardiac catheterization. Past 
medical history is significant for hypertension, type 2 diabetes mellitus, and 
hyperlipidemia."""

summarize_text(medical)

### Domain: Game Lore

In [ ]:
game_lore = """The Legend of Zelda series has captivated gamers for nearly four decades 
with its unique blend of exploration, puzzle-solving, and combat. Starting with the 
original 1986 NES title, the franchise established a template for action-adventure 
games that continues to influence game design today. Each entry reimagines the core 
formula while maintaining the series identity: a young hero named Link must rescue 
Princess Zelda and defeat the villain Ganondorf across a sprawling fantasy world. 
The 2017 release of Breath of the Wild revolutionized open-world game design by 
giving players complete freedom to explore Hyrule in any order."""

summarize_text(game_lore)

---
## Experiments

### Experiment 1: Test Your Own Domain

Pick a type of text and run it through both models. What domain is it from? How does each model handle it?

In [ ]:
# Experiment 1: Your own text
my_text = ""  # <-- Paste your text here

if my_text:
    print("=== TEXT GENERATOR ===")
    generate_from_prompt(my_text)
    
    print("=== SUMMARIZER ===")
    if len(my_text.split()) >= 30:
        summarize_text(my_text)
    else:
        print("(Text too short to summarize — needs at least 30 words)")
else:
    print("Paste some text between the quotes above and run this cell!")

### Experiment 2: Find the Domain That Breaks Both Models

Try to find text where BOTH models produce bad output. Ideas: mixing languages, heavy sarcasm, recipes, math problems, code, song lyrics.

In [ ]:
# Experiment 2: Break both models
breaking_text = ""  # <-- Paste text that you think will confuse both models

if breaking_text:
    print("=== TEXT GENERATOR ===")
    generate_from_prompt(breaking_text)
    
    print("=== SUMMARIZER ===")
    if len(breaking_text.split()) >= 30:
        summarize_text(breaking_text)
    else:
        print("(Text too short to summarize)")
else:
    print("Paste some text that you think will confuse both models!")

### Experiment 3: Domain Comparison Table

Run 3 domains through both models. Which model handles domain shift better?

| Domain | Generator Quality (1-5) | Summarizer Quality (1-5) | Notes |
|--------|------------------------|--------------------------|-------|
| News | | | |
| Poetry | | | |
| Medical | | | |
| Game | | | |
| Legal | | | |

---
## My Observations

Write your observations in this cell (double-click to edit):

- Which domains did the **text generator** handle best? Why?
- Which domains did the **summarizer** handle best? Why?
- Was there a domain where **both** models struggled? What made it hard?
- Can you connect the failures to what each model was **trained on**?

*Your answers here...*

---
## Vocabulary

| Term | Meaning |
|------|---------||
| **Domain** | A category of text (tweets, legal documents, poetry, medical notes, etc.) |
| **Overfitting** | When a model is too specialized in its training data to handle new situations |
| **Domain shift** | When the real-world data is different from the training data |
| **Generalization** | A model's ability to work well on data it hasn't seen before |
| **Distribution** | The patterns and characteristics of a particular type of text |
| **Pretraining** | Training a model on a massive general dataset before fine-tuning for a specific task |